# Dataset Inspection: dataset-labeled-anon-ip.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

DATA_PATH = '../../../data/cscas/dataset-labeled-anon-ip.csv'

## 1. Load & Basic Info

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['Timestamp'])
print(f'Shape: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
df.dtypes

In [ ]:
df.head()

In [ ]:
df['IntIP'].value_counts()

In [ ]:
df['ExtIP'].value_counts().get('-1', 0)

In [ ]:
df["AlertCount"].value_counts()

## 3. Label Distribution

In [ ]:
print('Label value counts:')
print(df['Label'].value_counts())
print(f'\nClass balance: {df["Label"].value_counts(normalize=True).round(4).to_dict()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

vc = df['Label'].value_counts()
axes[0].bar(vc.index.astype(str), vc.values)
axes[0].set_title('Label distribution (count)')
axes[0].set_xlabel('Label')

axes[1].pie(vc.values, labels=vc.index.astype(str), autopct='%1.1f%%')
axes[1].set_title('Label distribution (%)')

plt.tight_layout()
plt.show()

In [ ]:
print('SCAS value counts:')
print(df['SCAS'].value_counts())

## 4. Temporal Coverage

In [ ]:
print(f'Time range: {df["Timestamp"].min()} → {df["Timestamp"].max()}')
print(f'Duration: {df["Timestamp"].max() - df["Timestamp"].min()}')

In [ ]:
half_daily = df.set_index('Timestamp').resample('12h').agg(
    total=('Label', 'count'),
    attacks=('Label', lambda x: (x == 1).sum())
)

# Classify each signature by the label(s) it has ever appeared with, globally
sig_labels = df.groupby('SignatureID')['Label'].agg(lambda s: set(s))
sig_class = sig_labels.apply(
    lambda s: 'mixed' if len(s) > 1 else ('attack' if 1 in s else 'benign')
)
df['sig_class'] = df['SignatureID'].map(sig_class)

# Unique signatures seen per day, split by their global class
daily_sig_class = (
    df.set_index('Timestamp')
      .groupby([pd.Grouper(freq='D'), 'SignatureID'])['sig_class']
      .first()
      .reset_index()
)
daily_sig_counts = (
    daily_sig_class.groupby(['Timestamp', 'sig_class'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['benign', 'attack', 'mixed'], fill_value=0)
    .reindex(half_daily.index, fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 4))

ax2 = ax.twinx()
bar_kwargs = dict(width=1.0, alpha=0.35, linewidth=0)
ax2.bar(daily_sig_counts.index, daily_sig_counts['benign'],
        label='Benign-only signatures', color='tab:green', **bar_kwargs)
ax2.bar(daily_sig_counts.index, daily_sig_counts['attack'],
        bottom=daily_sig_counts['benign'],
        label='Attack-only signatures', color='tab:red', **bar_kwargs)
# ax2.bar(daily_sig_counts.index, daily_sig_counts['mixed'],
#         bottom=daily_sig_counts['benign'] + daily_sig_counts['attack'],
#         label='Mixed signatures', color='tab:orange', **bar_kwargs)
ax2.set_ylabel('# unique signatures')

# keep line plot drawn on top of the bars
ax.set_zorder(ax2.get_zorder() + 1)
ax.patch.set_visible(False)

ax.plot(half_daily.index, half_daily['total'], label='Total alerts', linewidth=0.8, color='tab:blue')
ax.plot(half_daily.index, half_daily['attacks'], label='Attack alerts (Label=1)', linewidth=0.8, color='black')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.set_title('CSCAS alert volume timeline (bin 12H) and ratio of unique signatures by class')
ax.set_ylabel('# alerts')
ax.set_yscale('log')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Signature Analysis

In [ ]:
print(f'Unique signatures: {df["SignatureID"].nunique()}')
print(f'Unique signature texts: {df["SignatureText"].nunique()}')

# all unique sigs to file for inspection
df[['SignatureID', 'SignatureText']].drop_duplicates().sort_values('SignatureID').to_csv('unique_signatures.csv', index=False) 

In [ ]:
top_sigs = df.groupby(['SignatureID', 'SignatureText']).agg(
    count=('Label', 'count'),
    attacks=('Label', 'sum'),
    attack_rate=('Label', 'mean')
).sort_values('count', ascending=False).head(20)

top_sigs['attack_rate%'] = (top_sigs['attack_rate'] * 100).round(3)
top_sigs[['count', 'attacks', 'attack_rate%']]

In [ ]:
# Top signatures by attack rate (min 10 alerts to filter noise)
all_sigs = df.groupby(['SignatureID', 'SignatureText']).agg(
    count=('Label', 'count'),
    attacks=('Label', 'sum'),
    attack_rate=('Label', 'mean')
).query('count >= 10').sort_values('attack_rate', ascending=False).head(20)

all_sigs['attack_rate%'] = (all_sigs['attack_rate'] * 100).round(1)
all_sigs[['count', 'attacks', 'attack_rate%']]

In [ ]:
# Top signatures by count, colored by attack rate
top20 = top_sigs.reset_index()
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top20['SignatureText'].str[:50], top20['count'], color=plt.cm.RdYlGn_r(top20['attack_rate']))
ax.set_xlabel('Alert count')
ax.set_title('Top 20 signatures by alert count (color = attack rate)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of signature frequency by label
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, grp in df.groupby('Label'):
    axes[0].hist(grp['SignatureID'], bins=50, alpha=0.5, density=True, label=f'Label={label}')
axes[0].set_title('Signature distribution by label')
axes[0].set_xlabel('SignatureID')
axes[0].legend()

plt.tight_layout()
plt.show()

## 6. Protocol & Port Distribution

In [ ]:
print('Protocol value counts (top 10):')
print(df['Proto'].value_counts().head(10))

## 7. Similarity Feature Overview

In [ ]:
# Replace -1 with NaN for stats on real similarity values
sim_df = df[sim_cols].replace(-1, np.nan)
sim_stats = sim_df.describe().T
sim_stats['coverage%'] = (sim_df.notna().mean() * 100).round(1)
sim_stats[['coverage%', 'mean', 'std', 'min', 'max']].sort_values('coverage%', ascending=False)

In [ ]:
# Coverage heatmap
coverage = sim_df.notna().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(coverage.index[::-1], coverage.values[::-1])
ax.set_xlabel('Coverage (fraction non-missing)')
ax.set_title('Similarity feature coverage')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of SCAS (aggregate similarity)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, grp in df.groupby('Label'):
    axes[0].hist(grp['Similarity'], bins=50, alpha=0.5, density=True, label=f'Label={label}')
axes[0].set_title('Similarity distribution by label')
axes[0].set_xlabel('Similarity')
axes[0].legend()

for label, grp in df.groupby('Label'):
    axes[1].hist(grp['SignatureMatchesPerDay'], bins=50, alpha=0.5, density=True, label=f'Label={label}')
axes[1].set_title('SignatureMatchesPerDay distribution by label')
axes[1].set_xlabel('SignatureMatchesPerDay')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Correlation of Similarity Features with Label

In [ ]:
corr_cols = sim_cols + ['Similarity', 'SignatureMatchesPerDay', 'AlertCount']
corr_with_label = df[corr_cols + ['Label']].replace(-1, np.nan).corr()['Label'].drop('Label').sort_values()

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['red' if v < 0 else 'steelblue' for v in corr_with_label.values]
ax.barh(corr_with_label.index, corr_with_label.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson correlation with Label')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.show()

## 9. Quick Sanity Checks

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('Timestamp is monotonic:', df['Timestamp'].is_monotonic_increasing)
print('Unique Label values:', sorted(df['Label'].unique()))
print('Unique SCAS values:', sorted(df['SCAS'].unique()))
print('IntIP unique count (anonymized):', df['IntIP'].nunique())
print('ExtIP unique count (anonymized):', df['ExtIP'].nunique())

In [ ]:
# Rows where IntIP == -1 (not captured)
print('Rows with IntIP == -1:', (df['IntIP'] == '-1').sum())

## 10. Tokenization

In [ ]:
from thesis.preprocessing.suricata_tokenization import tokenize_signatures_df

df_tok = pd.read_csv('unique_signatures.csv')
results = tokenize_signatures_df(df_tok)

df_out = pd.DataFrame([
    {
        "signature_id": s.signature_id,
        "ruleset": s.ruleset,
        "category": s.category,
        "description": s.description,
        "cve_refs": "|".join(sorted(s.cve_refs)),
        "qualifiers": "|".join(sorted(s.qualifiers)),
        "tokens": "|".join(sorted(s.tokens)),
    }
    for s in results
])

df_out.to_csv("tokenized_signatures2.csv", index=False)


## Do alert groups hit different IntIPS more often than just one?

In [ ]:
# df was overwritten by the tokenization step above (df = unique_signatures.csv); reload the original dataset
df = pd.read_csv(DATA_PATH, parse_dates=['Timestamp'])

In [ ]:
df['IntIP'].value_counts()

In [ ]:
# IntIP == -1 means the (signature, ExtIP) session touched more than one internal IP.
# If that's a small set of broad scanning signatures fanning out over many IPs (rather
# than genuinely diverse multi-signature behavior), these rows are low-value noise.
neg1 = df[df['IntIP'] == '-1']
print(f"Rows with IntIP == -1: {len(neg1)} ({len(neg1)/len(df):.1%} of all rows)")
print(f"Distinct signatures among IntIP == -1 rows: {neg1['SignatureID'].nunique()} (of {df['SignatureID'].nunique()} total)")

vc = neg1['SignatureID'].value_counts()
print(f"\nShare of IntIP=-1 rows covered by top 1 signature:  {vc.iloc[0] / len(neg1):.1%}")
print(f"Share of IntIP=-1 rows covered by top 5 signatures:  {vc.head(5).sum() / len(neg1):.1%}")
print(f"Share of IntIP=-1 rows covered by top 10 signatures: {vc.head(10).sum() / len(neg1):.1%}")

In [ ]:
sig_text = neg1.drop_duplicates('SignatureID').set_index('SignatureID')['SignatureText']
top15 = vc.head(15).rename('count').to_frame()
top15['share%'] = (top15['count'] / len(neg1) * 100).round(1)
top15['signature'] = top15.index.map(sig_text)
print(top15[['count', 'share%', 'signature']].to_string())

print(f"\nDistinct ExtIP among IntIP=-1 rows: {neg1['ExtIP'].nunique()}")
print(f"\nLabel distribution among IntIP=-1 rows:\n{neg1['Label'].value_counts()}")

## 11. Grouped Dataset — (IntIP, time window) binning (`cscas_target_window`)

Baskets are keyed by (internal target IP, 1h window) instead of one basket per CSV row/signature (`cscas_pregrouped`), so a basket can span multiple distinct signatures fired against the same target. Rows with `IntIP == -1` (session touched multiple internal IPs, not resolvable to one target) are excluded — see `group_cscas_rows_by_target_window` in `src/thesis/grouping/group_alerts.py`.

### Alert volume per internal IP over time

Before picking a time-window / session-gap scheme, look at how raw traffic against individual internal IPs is actually distributed over time — bursty with clear gaps, or roughly continuous? This uses the raw CSV rows (`df`), not the pre-built `cscas_target_window` baskets, since we're informing that design choice.

In [ ]:
# Heatmap: alert volume per (internal IP, hourly time bin), for the busiest IPs.
# IntIP == -1 excluded (not a single target -- see section 11 intro).
N_TOP_IPS = 25
BIN = '1h'

df_valid = df[df['IntIP'] != '-1'].copy()
top_ips = df_valid['IntIP'].value_counts().head(N_TOP_IPS).index.tolist()

ts_by_ip = (
    df_valid[df_valid['IntIP'].isin(top_ips)]
    .groupby(['IntIP', pd.Grouper(key='Timestamp', freq=BIN)])
    .size()
    .unstack('IntIP')
    .fillna(0)
    .reindex(columns=top_ips)  # keep busiest-first row order in the heatmap
)

fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(
    np.log1p(ts_by_ip.T.values),
    aspect='auto',
    cmap='viridis',
    interpolation='nearest',
)
ax.set_yticks(range(len(top_ips)))
ax.set_yticklabels(top_ips, fontsize=8)
ax.set_xlabel(f'time bin ({BIN})')
ax.set_title(f'Alert volume per internal IP over time (top {N_TOP_IPS} busiest IPs, log1p count)')
fig.colorbar(im, ax=ax, label='log1p(alert count)')
plt.tight_layout()
plt.show()

In [ ]:
ts_by_ip = (
    df_valid[df_valid['IntIP'].isin(top_ips)]
    .groupby(['IntIP', pd.Grouper(key='Timestamp', freq='6H')])
    .size()
    .unstack('IntIP')
    .fillna(0)
    .reindex(columns=top_ips)  # keep busiest-first row order in the heatmap
)

fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(
    np.log1p(ts_by_ip.T.values),
    aspect='auto',
    cmap='viridis',
    interpolation='nearest',
)
ax.set_yticks(range(len(top_ips)))
ax.set_yticklabels(top_ips, fontsize=8)
ax.set_xlabel(f'time bin (6H)')
ax.set_title(f'Alert volume per internal IP over time (top {N_TOP_IPS} busiest IPs, log1p count)')
fig.colorbar(im, ax=ax, label='log1p(alert count)')
plt.tight_layout()
plt.show()

In [ ]:
import json

In [ ]:
target_window_group_file = "../../../artifacts/cache/cscas/groups/cscas_target_window_w21600s/alert_groups/alert_groups_raw.json"

with open(target_window_group_file, 'r') as f:
    tw_groups = json.load(f)

print(f'Loaded {len(tw_groups)} target-window alert_groups')

In [ ]:
tw_df = pd.DataFrame([
    {
        'group_id': g['group_id'],
        'int_ip': g['int_ip'],
        'n_alerts': g['n_alerts'],
        'n_rows': len(g['sorted_items']),
        'n_items': len(g['abs_items']),
        'group_label': g['group_label'],
        'start_ts': g['start_ts'],
        'end_ts': g['end_ts'],
    }
    for g in tw_groups
])

print(f'Baskets: {len(tw_df)}')
print(f'Distinct int_ip: {tw_df["int_ip"].nunique()}')
print()
print('group_label distribution:')
print(tw_df['group_label'].value_counts())
print()
print('n_rows (CSV rows aggregated per basket) — describe:')
print(tw_df['n_rows'].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
tw_df['n_rows'].clip(upper=100).value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Basket size distribution (rows aggregated per (IntIP, 1h window) basket, clipped at 20)')
ax.set_xlabel('rows in basket')
ax.set_ylabel('count of baskets')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

### Are baskets actually multi-signature, or just the same signature repeated?

`n_rows > 1` alone doesn't mean a basket is interesting for co-occurrence mining — a target hit 11 times by the *same* signature in an hour is still a single-signature basket. Count distinct token-sets (i.e. distinct signatures) within each basket's `sorted_items`.

In [ ]:
def n_distinct_signatures(g):
    return len({tuple(sorted(s)) for s in g['sorted_items']})

tw_df['n_distinct_sigs'] = [n_distinct_signatures(g) for g in tw_groups]

multi_row = tw_df[tw_df['n_rows'] > 1]
diverse = multi_row[multi_row['n_distinct_sigs'] > 1]

print(f"Baskets with >1 row: {len(multi_row)} ({len(multi_row) / len(tw_df):.1%} of all baskets)")
print(f"Baskets genuinely spanning >1 distinct signature: {len(diverse)} ({len(diverse) / len(tw_df):.1%} of all baskets)")
print(f"  = {len(diverse) / len(multi_row):.1%} of multi-row baskets")
print()
print('Distribution of n_distinct_sigs among multi-row baskets:')
print(multi_row['n_distinct_sigs'].value_counts().sort_index().head(15))

### Cross-signature category co-occurrence

Which pairs of signature categories (`cat:EXPLOIT`, `cat:WEB_SERVER`, ...) actually co-occur against the same target within an hour? This is the pattern `cscas_pregrouped` structurally cannot surface — every basket there is one signature, so no itemset can ever contain two categories.

In [ ]:
from itertools import combinations
from collections import Counter

pair_counts = Counter()
for g in tw_groups:
    cats = set()
    for row_tokens in g['sorted_items']:
        cats.update(t for t in row_tokens if t.startswith('cat:'))
    if len(cats) >= 2:
        for pair in combinations(sorted(cats), 2):
            pair_counts[pair] += 1

n_total = len(tw_groups)
top_pairs = pd.DataFrame(
    [(a, b, c, c / n_total) for (a, b), c in pair_counts.most_common(20)],
    columns=['category_a', 'category_b', 'count', 'support'],
)
top_pairs

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
labels = [f"{a} + {b}" for a, b in zip(top_pairs['category_a'], top_pairs['category_b'])]
ax.barh(labels[::-1], top_pairs['support'].values[::-1])
ax.set_xlabel('Support (fraction of baskets)')
ax.set_title('Top 20 cross-signature category co-occurrences (cscas_target_window)')
plt.tight_layout()
plt.show()

In [ ]:
# Eyeball one genuinely multi-signature basket
diverse_groups = [g for g in tw_groups if n_distinct_signatures(g) > 1]
sample = diverse_groups[0]

print('int_ip:', sample['int_ip'])
print('label:', sample['group_label'])
print('n_rows:', len(sample['sorted_items']))
print('abs_items (union):', sorted(sample['abs_items']))
print()
print('sorted_items (one row per original signature, timestamp order):')
for row_tokens in sample['sorted_items']:
    print(' -', sorted(row_tokens))

## 15. Dataset-level EDA (run_eda / run_eda_host parity)

Reproduces the analysis + plots from the old `run_eda.py` script for the `cscas` dataset (a single pre-grouped scenario, also named `cscas`), via `thesis.data.eda` (computation) and `thesis.visualization.eda` (plotting). No per-host section here -- cscas rows have no `host` column, that analysis is AIT-ADS only. Output is written under `artifacts/experiments/run_eda/cscas/`, same layout the script used.

In [ ]:
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO2 = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO2 / 'src'))

import thesis.data.eda as de
from thesis.configs import load_scenarios
from thesis.visualization.eda import load_alerts, plot_label_distribution_table

DATASET2 = 'cscas'
DATA_DIR2 = REPO2 / 'data' / 'cscas'
scenarios2 = load_scenarios(DATASET2)
print(f'Scenarios: {scenarios2}')

run_ts2 = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
run_dir2 = REPO2 / 'artifacts' / 'experiments' / 'run_eda' / DATASET2 / f'run_{run_ts2}_raw'
summary_dir2 = run_dir2 / 'summary'
print(f'Output dir: {run_dir2}')

### Load alerts

In [ ]:
all_df2 = load_alerts(str(DATA_DIR2), scenarios=scenarios2, dataset=DATASET2)
print(f'{len(all_df2):,} alerts loaded.')
all_df2.head()

### Phase 1: per-scenario analysis

cscas is pre-grouped (one row per already-aggregated basket, not a raw alert-by-alert stream), so alert_group pair-frequency analysis doesn't apply here (`skip_grouping=True`) -- this just writes the per-scenario text summary (basic stats, missing values, label distribution, top signatures).

In [ ]:
for scenario in scenarios2:
    scenario_df2 = all_df2[all_df2['scenario'] == scenario].copy()
    de.run_scenario_eda(
        scenario_df2,
        scenario,
        out_path=run_dir2 / scenario,
        summary_path=summary_dir2,
        alert_groups_base_dir=REPO2 / 'artifacts' / 'alert_groups',
        skip_grouping=True,
    )

### Cross-scenario tables

cscas has a single scenario, so these tables have one row -- still useful as the same standardized summary the AIT-ADS notebook produces.

In [ ]:
overview_df2 = de.compute_overview_table(all_df2)
overview_df2.to_csv(run_dir2 / 'overview_table.csv', index=False)
overview_df2

In [ ]:
label_dist_df2 = de.compute_label_distribution_table(all_df2)
label_dist_df2.to_csv(run_dir2 / 'label_distribution_table.csv', index=False)

fig, _ = plot_label_distribution_table(
    label_dist_df2, out_path=str(run_dir2 / 'label_distribution_table.png')
)

### Phase 2: overview plots

cscas-specific plots (temporal attack overview, signature event raster, signature purity pie) plus the shared overview plots (class balance, attack-type heatmap, top signatures, inter-arrival CDF, group size distribution, scenario overview table). No alert_group volume plots -- cscas skipped alert_group construction above.

In [ ]:
plots2 = de.build_overview_plots(scenarios2, all_df2, dataset=DATASET2, groups_df=None)
de.save_overview_plots(plots2, run_dir2 / 'plots', de.data_label(), close=False)

## 16. Field/Feature Overview — Summary CSVs

Tabular numerical overview of the raw dataset's fields, written to `overview_csvs/` next to this notebook: signature behavior (always-benign / always-attack / mixed, CVE coverage, per-day and per-hour activity), and network fields (IP/port/protocol cardinality and the `-1` "multiple values aggregated into this row" sentinel). Reloads `df` fresh from `DATA_PATH` so it's independent of whatever the preceding sections left `df` as.

In [ ]:
from pathlib import Path

df = pd.read_csv(DATA_PATH, parse_dates=['Timestamp'])

OVERVIEW_DIR = Path('overview_csvs')
OVERVIEW_DIR.mkdir(exist_ok=True)
print(f'Writing overview CSVs to {OVERVIEW_DIR.resolve()}')
print(f'{len(df):,} rows')

### 16.1 Signature analysis

Per-signature stats (count, benign/attack split, always-benign / always-attack / mixed classification, days/hours active, CVE refs), plus dataset-level rollups.

In [ ]:
sig_stats = df.groupby(['SignatureID', 'SignatureText']).agg(
    count=('Label', 'count'),
    attack_count=('Label', 'sum'),
).reset_index()
sig_stats['benign_count'] = sig_stats['count'] - sig_stats['attack_count']
sig_stats['attack_rate%'] = (sig_stats['attack_count'] / sig_stats['count'] * 100).round(3)
sig_stats['sig_class'] = np.select(
    [sig_stats['attack_count'] == 0, sig_stats['benign_count'] == 0],
    ['always_benign', 'always_attack'],
    default='mixed',
)

span = df.groupby('SignatureID').agg(
    n_days_active=('Timestamp', lambda s: s.dt.normalize().nunique()),
    n_hours_active=('Timestamp', lambda s: s.dt.floor('h').nunique()),
    first_seen=('Timestamp', 'min'),
    last_seen=('Timestamp', 'max'),
).reset_index()
sig_stats = sig_stats.merge(span, on='SignatureID')

# Attach CVE refs / category from the tokenization output (section 10), if it has been run.
try:
    tok = pd.read_csv('tokenized_signatures2.csv')
    sig_stats = sig_stats.merge(
        tok[['signature_id', 'cve_refs', 'category']].rename(columns={'signature_id': 'SignatureID'}),
        on='SignatureID', how='left',
    )
except FileNotFoundError:
    sig_stats['cve_refs'] = None
    sig_stats['category'] = None

sig_stats = sig_stats.sort_values('count', ascending=False)
sig_stats.to_csv(OVERVIEW_DIR / 'signature_overview.csv', index=False)
print(f'{len(sig_stats)} signatures -> signature_overview.csv')
sig_stats.head()

In [ ]:
all_cves = set()
if sig_stats['cve_refs'].notna().any():
    for refs in sig_stats['cve_refs'].dropna():
        if refs:
            all_cves.update(refs.split('|'))

signature_summary = pd.DataFrame([{
    'n_unique_signatures': sig_stats['SignatureID'].nunique(),
    'n_always_benign_signatures': int((sig_stats['sig_class'] == 'always_benign').sum()),
    'n_always_attack_signatures': int((sig_stats['sig_class'] == 'always_attack').sum()),
    'n_mixed_signatures': int((sig_stats['sig_class'] == 'mixed').sum()),
    'n_unique_cves': len(all_cves),
}])
signature_summary.to_csv(OVERVIEW_DIR / 'signature_summary.csv', index=False)
signature_summary

### 16.2 Signature activity per day / per hour

For each calendar-day and each hourly bin: how many distinct signatures were active, split by their global class (`always_benign` / `always_attack` / `mixed`), and what fraction of that bin's active signatures each class represents. `signature_activity_summary.csv` collapses each series to min/max/mean.

In [ ]:
sig_class_map = sig_stats.set_index('SignatureID')['sig_class']
df['_sig_overview_class'] = df['SignatureID'].map(sig_class_map)


def signature_activity_by_bin(freq):
    binned = df[['Timestamp', 'SignatureID', '_sig_overview_class']].copy()
    binned['bin'] = binned['Timestamp'].dt.floor(freq)
    active = binned.groupby(['bin', 'SignatureID'])['_sig_overview_class'].first().reset_index()
    counts = (
        active.groupby(['bin', '_sig_overview_class'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=['always_benign', 'always_attack', 'mixed'], fill_value=0)
    )
    counts['n_active_signatures'] = counts.sum(axis=1)
    counts['frac_benign'] = counts['always_benign'] / counts['n_active_signatures']
    counts['frac_attack'] = counts['always_attack'] / counts['n_active_signatures']
    counts['frac_mixed'] = counts['mixed'] / counts['n_active_signatures']
    return counts.reset_index().rename(columns={'bin': 'Timestamp'})


daily_sig_activity = signature_activity_by_bin('D')
hourly_sig_activity = signature_activity_by_bin('h')

daily_sig_activity.to_csv(OVERVIEW_DIR / 'signature_activity_per_day.csv', index=False)
hourly_sig_activity.to_csv(OVERVIEW_DIR / 'signature_activity_per_hour.csv', index=False)

print(f'{len(daily_sig_activity)} daily bins -> signature_activity_per_day.csv')
print(f'{len(hourly_sig_activity)} hourly bins -> signature_activity_per_hour.csv')
daily_sig_activity.head()

In [ ]:
def summarize_activity(counts_df, granularity):
    cols = ['always_benign', 'always_attack', 'mixed', 'n_active_signatures',
            'frac_benign', 'frac_attack', 'frac_mixed']
    return [
        {
            'granularity': granularity,
            'metric': col,
            'min': counts_df[col].min(),
            'max': counts_df[col].max(),
            'mean': counts_df[col].mean(),
        }
        for col in cols
    ]


signature_activity_summary = pd.DataFrame(
    summarize_activity(daily_sig_activity, 'day') + summarize_activity(hourly_sig_activity, 'hour')
)
signature_activity_summary[['min', 'max', 'mean']] = signature_activity_summary[['min', 'max', 'mean']].round(4)
signature_activity_summary.to_csv(OVERVIEW_DIR / 'signature_activity_summary.csv', index=False)
signature_activity_summary

### 16.3 Network fields — IP / port / protocol

Cardinality of `IntIP`, `ExtIP`, `Proto`, `IntPort`, `ExtPort`, plus how often each field's `-1` sentinel appears (meaning: the underlying session/basket touched *multiple* distinct values for that field, not resolvable to one).

In [ ]:
def multi_row_stats(series, sentinel):
    is_multi = series == sentinel
    return int(is_multi.sum()), round(float(is_multi.mean()) * 100, 3)


n_multi_int_ip, pct_multi_int_ip = multi_row_stats(df['IntIP'], '-1')
n_multi_ext_ip, pct_multi_ext_ip = multi_row_stats(df['ExtIP'], '-1')
n_multi_proto, pct_multi_proto = multi_row_stats(df['Proto'], -1)
n_multi_int_port, pct_multi_int_port = multi_row_stats(df['IntPort'], -1)
n_multi_ext_port, pct_multi_ext_port = multi_row_stats(df['ExtPort'], -1)

network_overview = pd.DataFrame([{
    'total_rows': len(df),
    'n_distinct_int_ip': df.loc[df['IntIP'] != '-1', 'IntIP'].nunique(),
    'n_rows_multi_int_ip': n_multi_int_ip,
    'pct_rows_multi_int_ip': pct_multi_int_ip,
    'n_distinct_ext_ip': df.loc[df['ExtIP'] != '-1', 'ExtIP'].nunique(),
    'n_rows_multi_ext_ip': n_multi_ext_ip,
    'pct_rows_multi_ext_ip': pct_multi_ext_ip,
    'n_distinct_protocols': df.loc[df['Proto'] != -1, 'Proto'].nunique(),
    'n_rows_multi_proto': n_multi_proto,
    'pct_rows_multi_proto': pct_multi_proto,
    'n_distinct_int_ports': df.loc[df['IntPort'] != -1, 'IntPort'].nunique(),
    'n_rows_multi_int_port': n_multi_int_port,
    'pct_rows_multi_int_port': pct_multi_int_port,
    'n_distinct_ext_ports': df.loc[df['ExtPort'] != -1, 'ExtPort'].nunique(),
    'n_rows_multi_ext_port': n_multi_ext_port,
    'pct_rows_multi_ext_port': pct_multi_ext_port,
}])
network_overview.to_csv(OVERVIEW_DIR / 'network_field_overview.csv', index=False)
network_overview.T

In [ ]:
proto_dist = df.groupby('Proto').agg(
    count=('Label', 'count'),
    attack_count=('Label', 'sum'),
).reset_index()
proto_dist['pct_of_rows'] = (proto_dist['count'] / len(df) * 100).round(3)
proto_dist['attack_rate%'] = (proto_dist['attack_count'] / proto_dist['count'] * 100).round(3)
proto_dist = proto_dist.sort_values('count', ascending=False)
proto_dist.to_csv(OVERVIEW_DIR / 'protocol_distribution.csv', index=False)
proto_dist

In [ ]:
def top_ports(col, n=25):
    sub = df[df[col] != -1]
    t = sub.groupby(col).agg(
        count=('Label', 'count'),
        attack_count=('Label', 'sum'),
    ).sort_values('count', ascending=False).head(n).reset_index()
    t['pct_of_rows'] = (t['count'] / len(df) * 100).round(4)
    t['attack_rate%'] = (t['attack_count'] / t['count'] * 100).round(3)
    return t


top_int_ports = top_ports('IntPort')
top_ext_ports = top_ports('ExtPort')
top_int_ports.to_csv(OVERVIEW_DIR / 'top_int_ports.csv', index=False)
top_ext_ports.to_csv(OVERVIEW_DIR / 'top_ext_ports.csv', index=False)
print('Top 10 internal ports:')
print(top_int_ports.head(10))

### 16.3b Port bucket distribution

Ports classified into IANA-style ranges: `well_known (0-1023)`, `registered (1024-49151)`, `ephemeral (49152-65535)`, plus the `-1` "multiple ports aggregated into this row" sentinel.

In [ ]:
def port_bucket_series(port_col):
    conditions = [
        port_col == -1,
        port_col < 1024,
        port_col < 49152,
    ]
    choices = ['multiple (-1)', 'well_known (0-1023)', 'registered (1024-49151)']
    return pd.Series(np.select(conditions, choices, default='ephemeral (49152-65535)'), index=port_col.index)


BUCKET_ORDER = ['well_known (0-1023)', 'registered (1024-49151)', 'ephemeral (49152-65535)', 'multiple (-1)']


def port_bucket_distribution(col):
    bucket = port_bucket_series(df[col])
    dist = (
        pd.DataFrame({'bucket': bucket, 'Label': df['Label']})
        .groupby('bucket')
        .agg(count=('Label', 'count'), attack_count=('Label', 'sum'))
        .reindex(BUCKET_ORDER, fill_value=0)
    )
    dist['pct_of_rows'] = (dist['count'] / len(df) * 100).round(3)
    dist['attack_rate%'] = (dist['attack_count'] / dist['count'] * 100).round(3)
    return dist.reset_index()


port_bucket_dist = pd.concat([
    port_bucket_distribution('IntPort').assign(port_type='internal'),
    port_bucket_distribution('ExtPort').assign(port_type='external'),
], ignore_index=True)
port_bucket_dist = port_bucket_dist[['port_type', 'bucket', 'count', 'attack_count', 'pct_of_rows', 'attack_rate%']]
port_bucket_dist.to_csv(OVERVIEW_DIR / 'port_bucket_distribution.csv', index=False)

print(port_bucket_dist.to_string(index=False))

### 16.4 Dataset-level rollup

One row tying the sections above together — the single table to skim first.

In [ ]:
dataset_overview = pd.DataFrame([{
    'total_rows': len(df),
    'start': df['Timestamp'].min(),
    'end': df['Timestamp'].max(),
    'duration_days': round((df['Timestamp'].max() - df['Timestamp'].min()).total_seconds() / 86400, 1),
    'n_benign_rows': int((df['Label'] == 0).sum()),
    'n_attack_rows': int((df['Label'] == 1).sum()),
    'attack_rate%': round((df['Label'] == 1).mean() * 100, 3),
    'n_unique_signatures': signature_summary['n_unique_signatures'].iloc[0],
    'n_always_benign_signatures': signature_summary['n_always_benign_signatures'].iloc[0],
    'n_always_attack_signatures': signature_summary['n_always_attack_signatures'].iloc[0],
    'n_mixed_signatures': signature_summary['n_mixed_signatures'].iloc[0],
    'n_unique_cves': signature_summary['n_unique_cves'].iloc[0],
    'n_distinct_int_ip': network_overview['n_distinct_int_ip'].iloc[0],
    'n_distinct_ext_ip': network_overview['n_distinct_ext_ip'].iloc[0],
    'n_distinct_protocols': network_overview['n_distinct_protocols'].iloc[0],
    'n_distinct_int_ports': network_overview['n_distinct_int_ports'].iloc[0],
    'n_distinct_ext_ports': network_overview['n_distinct_ext_ports'].iloc[0],
}])
dataset_overview.to_csv(OVERVIEW_DIR / 'dataset_overview.csv', index=False)
dataset_overview.T

In [ ]:
print('Generated overview CSVs:')
for f in sorted(OVERVIEW_DIR.glob('*.csv')):
    print(' -', f.name)

## 17. Contrast-Set Growth-Rate Distribution

`min_growth_rate` (attribute mining's Step 1 threshold, see
`thesis.mining.attribute_contrast_mining`) currently gets swept over
`{3.0, 4.0, 5.0}` in `screening_mining_settings.yaml` and
`sweep_attribute_schema.sh` -- values chosen without ever looking at what
growth rates the CSCAS data actually produces. `parameter_importance` in
`config_selection.ipynb` found `min_growth_rate` had essentially zero effect
on model precision (eta²≈0.0002); before treating that as "this threshold
doesn't matter," check whether `{3.0, 4.0, 5.0}` even spans a meaningfully
different region of the real distribution.

`growth_rate` (`confidence_attack / confidence_benign` for a candidate
predicate) is computed once per candidate by
`compute_predicate_contrast_stats`, for every single and pairwise
categorical predicate, **before** any `min_growth_rate`/coverage threshold
is applied -- filtering happens later, in the separate
`filter_contrast_survivors`, which this section never calls. So this runs
directly against the whole `cscas_pregrouped` dataset (not one mining-sweep
run's leftover window), independent of any prior mining-sweep run having
happened.

In [ ]:
from thesis.grouping.group_alerts import CSCAS_PREGROUPED_METHOD
from thesis.mining.attribute_contrast_mining import (
    build_categorical_predicate_matrix,
    compute_predicate_contrast_stats,
)
from thesis.paths import CACHE_DIR
from thesis.pipeline.pipeline import ingest_cscas_scenario, load_or_build_alert_groups

# Same cache_dir convention mine_attribute_schema.py uses for cscas
# (thesis/scripts/mining/mine_attribute_schema.py's mine_scenario).
cscas_cache_dir = CACHE_DIR / "cscas" / "groups" / CSCAS_PREGROUPED_METHOD
ingest_cscas_scenario(cache_dir=cscas_cache_dir)
cscas_alert_groups = load_or_build_alert_groups("cscas", cscas_cache_dir)
print(f"Loaded {len(cscas_alert_groups)} cscas_pregrouped alert_groups")

X_cat, X_num, y_ag, column_predicate_map = build_categorical_predicate_matrix(cscas_alert_groups)
contrast_stats = compute_predicate_contrast_stats(X_cat, y_ag, column_predicate_map)
print(f"{len(contrast_stats)} candidate predicates (single + pairwise), before any threshold filter")
contrast_stats[["itemset", "growth_rate", "confidence_attack", "confidence_benign", "support_count"]].describe()

In [ ]:
CURRENT_THRESHOLDS = [3.0, 4.0, 5.0]  # the existing screening_mining_settings.yaml / sweep_attribute_schema.sh grid

# growth_rate == 0 means the predicate never fires on any attack alert_group
# (attack_support == 0) -- not a candidate contrast-mining could ever surface
# regardless of threshold, and log(0) isn't plottable. Report it separately,
# exclude it from the distribution below.
gr = contrast_stats["growth_rate"]
n_zero = int((gr == 0).sum())
gr_nonzero = gr[gr > 0]
print(f"{n_zero}/{len(gr)} candidates never fire on any attack (growth_rate=0, excluded below)")
print(f"{len(gr_nonzero)} candidates with growth_rate > 0")

fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.logspace(np.log10(gr_nonzero.min()), np.log10(gr_nonzero.max()), 40)
ax.hist(gr_nonzero, bins=bins, color="#4477AA")
ax.set_xscale("log")
ax.set_xlabel("growth_rate (log scale)")
ax.set_ylabel("# candidate predicates")
ax.set_title(f"cscas: growth_rate distribution across {len(gr_nonzero)} candidate predicates (Step 1, unfiltered)")

palette = ["#CC3311", "#EE7733", "#228833"]
for t, color in zip(CURRENT_THRESHOLDS, palette):
    pct = (gr >= t).mean() * 100
    ax.axvline(t, color=color, linestyle="--", linewidth=1.5, label=f"min_growth_rate={t:g} ({pct:.1f}% of all candidates survive)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# Finer-grained survival table, including thresholds well below the current
# grid, to see where the real "elbow" (the point candidate survival actually
# drops sharply) sits relative to {3.0, 4.0, 5.0}.
survival = pd.DataFrame(
    [
        {"threshold": t, "n_survive": int((gr >= t).sum()), "pct_of_all": (gr >= t).mean() * 100}
        for t in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 8.0, 10.0, 50.0]
    ]
)
survival

**Reading this**: if the survival percentage barely moves between `min_growth_rate=3.0` and `5.0` in the table above, that's a direct, data-level explanation for `config_selection.ipynb`'s eta²≈0.0002 finding — the swept range doesn't select meaningfully different candidate sets, so it can't have moved the downstream model either. If instead survival drops sharply somewhere well below `3.0` (e.g. near `1.0`–`1.5`, the neutral point where a predicate stops being attack-indicative at all), that's the region actually worth screening -- both for a future revision of `screening_mining_settings.yaml`'s grid, and as the `min_growth_rate` value to hold fixed for the `class_weight`/`min_samples_leaf` sweep points below (`sweep_attribute_schema.sh`).

## 18. Raw Field Discriminativeness (Categorical, Numerical, Mutual Information)

Section 17 covers `growth_rate`, computed per *predicate value* once contrast-set
mining has already enumerated candidates. This section steps one level up: for
each raw *field* on an `AlertGroup` (before any mining, one-hot expansion, or
threshold), how much signal does it carry about the label on its own?

- **Categorical fields** (`category`, `ruleset`, `proto`, `scas`, `cve_present`,
  ...): for each value `v` a field can take, the local attack rate
  `p_v = P(attack | field=v)` compared against the dataset base rate, summarized
  per-field by **Cramér's V** (a chi-square-derived association score, 0-1,
  comparable across fields regardless of how many values each takes).
- **Numeric fields** (`similarity`, `signature_matches_per_day`, `cve_age_years`,
  ...): **point-biserial correlation** (Pearson correlation against the binary
  label) and **AUC-based separability** (the field's raw value used directly as
  a 1-D classifier score) -- two complementary reads on the same relationship.
- **Mutual information** `I(field; Label)` unifies both: an encoding-agnostic
  measure that captures any statistical dependency, not only the
  monotonic/single-threshold relationships the measures above are each
  individually sensitive to.

All four are implemented in `thesis.metrics.field_discriminativeness` and computed
directly on the raw fields via `compute_candidate_attribute_features` (the same
per-`AlertGroup` extractor mining itself uses, reused here rather than
re-derived, via `thesis.mining.attribute_features` -- so this EDA never drifts
out of sync with what mining actually consumes).

These stats characterize the *raw data*, independent of any mining
configuration -- they cannot say what a particular mining strategy, with its
own filters and thresholds, will ultimately extract. The interpretation cell at
the end flags which raw fields look most promising ahead of mining, to be
revisited later against the mining pipeline's actual output.

In [ ]:
from thesis.mining.attribute_features import (
    BINARY_CATEGORICAL_FIELDS,
    MULTI_VALUED_CATEGORICAL_FIELDS,
    NUMERIC_FIELDS,
    compute_candidate_attribute_features,
)
from thesis.schemas.preprocessing import ATTR_SIMILARITY_COLUMNS

# The "core" interpretable fields for the headline plots below -- the
# namespaced qualifier_*/attr_populated:*/attr_value:*/applicable_layer:*
# expansions in BINARY_CATEGORICAL_FIELDS/NUMERIC_FIELDS are left to the full
# mutual-information table further down instead, to keep these plots readable.
CORE_CATEGORICAL_FIELDS = list(MULTI_VALUED_CATEGORICAL_FIELDS) + [
    "cve_present", "multi_target", "multi_ext_port", "multi_int_port", "proto_mismatch",
]
CORE_NUMERIC_FIELDS = [
    "signature_matches_per_day", "alert_count", "similarity", "signature_id_similarity", "cve_age_years",
]

# -1.0 is this codebase's "not applicable" sentinel for numeric attribute
# features (thesis.mining.attribute_features._NOT_APPLICABLE) -- only
# cve_age_years and the attr_value:* columns actually use it; the other
# numeric fields default to 0.0 (a real value), not -1.
NUMERIC_SENTINELS = {
    "cve_age_years": -1.0,
    **{f"attr_value:{name}": -1.0 for name in ATTR_SIMILARITY_COLUMNS},
}

# One pass over cscas_alert_groups (loaded in section 17 above), reusing the
# same single-source-of-truth extractor mining itself calls.
field_rows = [compute_candidate_attribute_features(tx) for tx in cscas_alert_groups]
field_df = pd.DataFrame(field_rows)
field_df["Label"] = [1 if tx.group_label == "attack" else 0 for tx in cscas_alert_groups]

print(f"{len(field_df)} alert_groups, base attack rate {field_df['Label'].mean() * 100:.2f}%")
field_df[CORE_CATEGORICAL_FIELDS + CORE_NUMERIC_FIELDS].describe(include="all").T

In [ ]:
from IPython.display import display

from thesis.metrics.field_discriminativeness import cramers_v, local_attack_rates

cramers_v_scores = pd.Series(
    {field: cramers_v(field_df[field], field_df["Label"]) for field in CORE_CATEGORICAL_FIELDS}
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(cramers_v_scores.index, cramers_v_scores.values, color="#4477AA")
ax.set_xlabel("Cramér's V")
ax.set_title("Categorical field discriminativeness (Cramér's V vs. Label)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Local attack rate vs. base rate for the two strongest fields' values.
for field in cramers_v_scores.index[:2]:
    rates = local_attack_rates(field_df, field, "Label")
    print(f"\n{field} (Cramér's V={cramers_v_scores[field]:.3f}, base rate {rates['base_rate'].iloc[0]:.3f}):")
    display(rates.head(10)[["value", "support", "attack_rate", "deviation"]])

In [ ]:
from thesis.metrics.field_discriminativeness import auc_separability, point_biserial_score

numeric_rows = []
for field in CORE_NUMERIC_FIELDS:
    sentinel = NUMERIC_SENTINELS.get(field)
    r, p = point_biserial_score(field_df[field], field_df["Label"], sentinel=sentinel)
    auc = auc_separability(field_df[field], field_df["Label"], sentinel=sentinel)
    numeric_rows.append({"field": field, "point_biserial_r": r, "p_value": p, "auc_separability": auc})
numeric_scores = pd.DataFrame(numeric_rows).sort_values("auc_separability", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

colors = ["red" if v < 0 else "steelblue" for v in numeric_scores["point_biserial_r"]]
axes[0].barh(numeric_scores["field"], numeric_scores["point_biserial_r"], color=colors)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Point-biserial correlation with Label")
axes[0].set_xlabel("r")

axes[1].barh(numeric_scores["field"], numeric_scores["auc_separability"], color="#4477AA")
axes[1].axvline(0.5, color="black", linewidth=0.8, linestyle="--", label="no separation (0.5)")
axes[1].set_title("AUC-based separability")
axes[1].set_xlabel("max(AUC, 1-AUC)")
axes[1].set_xlim(0.4, 1.0)
axes[1].legend(fontsize=8)

for ax in axes:
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

numeric_scores

In [ ]:
from thesis.metrics.field_discriminativeness import field_discriminativeness_table

# The full field set this time -- including the namespaced qualifier_*/
# attr_populated:*/attr_value:*/applicable_layer:* expansions skipped above.
ALL_CATEGORICAL_FIELDS = list(MULTI_VALUED_CATEGORICAL_FIELDS) + list(BINARY_CATEGORICAL_FIELDS)
ALL_NUMERIC_FIELDS = list(NUMERIC_FIELDS)

mi_table = field_discriminativeness_table(
    field_df,
    categorical_fields=ALL_CATEGORICAL_FIELDS,
    numeric_fields=ALL_NUMERIC_FIELDS,
    label_col="Label",
    sentinels=NUMERIC_SENTINELS,
)
print(f"{len(mi_table)} raw fields ranked by mutual information with Label")

top_mi = mi_table.head(30)
fig, ax = plt.subplots(figsize=(9, 10))
colors = ["#CC3311" if t == "categorical" else "#4477AA" for t in top_mi["field_type"]]
ax.barh(top_mi["field"], top_mi["mutual_information"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("Mutual information I(field; Label)")
ax.set_title(f"Top 30 of {len(mi_table)} raw fields by mutual information (red=categorical, blue=numeric)")
plt.tight_layout()
plt.show()

mi_table.head(20)

**Reading this**: `mutual_information` is the one score comparable across
categorical and numeric fields alike, so it's the ranking to lead with; Cramér's
V / point-biserial r / AUC separability explain *how* a given field is
informative (association strength, direction, and single-threshold
separability, respectively) once MI has flagged it as worth a closer look. The
fields at the top of this ranking are exactly the raw signal
`compute_predicate_contrast_stats` (section 17) and the decision-tree rule
extraction step have to work with -- **revisit this ranking later against the
mining pipeline's actual output** (e.g. `attribute_mining_sweep_eda.ipynb`'s
mined-schema composition) to see how well raw-field discriminativeness
anticipated what mining actually kept.